In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

In [3]:
from src.volatility import (
    add_close_to_close_volatility,
    add_parkinson_volatility,
    add_garman_klass_volatility
)

from src.labeling import add_returns
from src.features import (
    add_target_direction,
    add_lagged_returns,
    add_moving_average_features,
    add_volatility_features
)

In [4]:
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df.set_index("Date", inplace=True)

df = add_close_to_close_volatility(df)
df = add_parkinson_volatility(df)
df = add_garman_klass_volatility(df)
df.tail()

,Open,High,Low,Close,Volume,vol_cc,vol_pk,vol_gk
Date,,,,,,,,
2026-02-13,25571.150391,25630.349609,25444.300781,25471.099609,453500,0.138191,0.109934,0.106763
2026-02-16,25423.599609,25697.000000,25372.699219,25682.750000,275800,0.141515,0.111864,0.107197
2026-02-17,25637.949219,25764.400391,25570.300781,25725.400391,344100,0.140728,0.112240,0.107636
2026-02-18,25752.650391,25828.050781,25645.150391,25819.349609,310200,0.130728,0.107796,0.105401
2026-02-19,25873.349609,25885.300781,25388.750000,25454.349609,0,0.141139,0.110805,0.103567


In [5]:
def make_regime(series):
    q1 = series.quantile(0.33)
    q2 = series.quantile(0.66)

    return pd.cut(
        series,
        bins=[-np.inf, q1, q2, np.inf],
        labels=["Low","Medium","High"]
    )

df["regime_cc"] = make_regime(df["vol_cc"])
df["regime_pk"] = make_regime(df["vol_pk"])
df["regime_gk"] = make_regime(df["vol_gk"])

In [6]:
df = add_returns(df)

df = add_target_direction(df)
df = add_lagged_returns(df)
df = add_moving_average_features(df)
df = add_volatility_features(df, vol_col="vol_cc")
df = add_volatility_features(df, vol_col="vol_pk")
df = add_volatility_features(df, vol_col="vol_gk")

df_feat = df.dropna().copy()

In [7]:
feature_sets = {
    "cc": [
        "ret_lag_1","ret_lag_5","ret_lag_10",
        "ma_ratio_5","ma_ratio_10","ma_ratio_20",
        "vol_cc","vol_cc_lag_1"
    ],
    "pk": [
        "ret_lag_1","ret_lag_5","ret_lag_10",
        "ma_ratio_5","ma_ratio_10","ma_ratio_20",
        "vol_pk","vol_pk_lag_1"
    ],
    "gk": [
        "ret_lag_1","ret_lag_5","ret_lag_10",
        "ma_ratio_5","ma_ratio_10","ma_ratio_20",
        "vol_gk","vol_gk_lag_1"
    ]
}

In [8]:
split_idx = int(len(df_feat) * 0.8)

train = df_feat.iloc[:split_idx]
test  = df_feat.iloc[split_idx:]

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def evaluate_regime(regime_col, feature_cols):

    X_train = train[feature_cols]
    y_train = train["target"]

    X_test = test[feature_cols]
    y_test = test["target"]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_s, y_train)

    pred = model.predict(X_test_s)

    overall_acc = accuracy_score(y_test, pred)

    test_eval = test.copy()
    test_eval["pred"] = pred

    regime_acc = (
        test_eval
        .groupby(regime_col)
        .apply(lambda d: accuracy_score(d["target"], d["pred"]))
    )

    return overall_acc, regime_acc

In [10]:
acc_cc, reg_cc = evaluate_regime("regime_cc", feature_sets["cc"])
acc_pk, reg_pk = evaluate_regime("regime_pk", feature_sets["pk"])
acc_gk, reg_gk = evaluate_regime("regime_gk", feature_sets["gk"])

In [11]:
print("Overall Accuracy")
print("Close-Close :", acc_cc)
print("Parkinson   :", acc_pk)
print("Garman-Klass:", acc_gk)

print("\nRegime Accuracy — CC")
print(reg_cc)

print("\nRegime Accuracy — PK")
print(reg_pk)

print("\nRegime Accuracy — GK")
print(reg_gk)

Overall Accuracy
Close-Close : 0.5444444444444444
Parkinson   : 0.5411111111111111
Garman-Klass: 0.5422222222222223

Regime Accuracy — CC
regime_cc
Low       0.545139
Medium    0.526971
High      0.590361
dtype: float64

Regime Accuracy — PK
regime_pk
Low       0.549902
Medium    0.520325
High      0.700000
dtype: float64

Regime Accuracy — GK
regime_gk
Low       0.551587
Medium    0.520216
High      0.680000
dtype: float64


In [12]:
print("CC regime counts")
print(test["regime_cc"].value_counts())

print("\nPK regime counts")
print(test["regime_pk"].value_counts())

print("\nGK regime counts")
print(test["regime_gk"].value_counts())

CC regime counts
regime_cc
Low       576
Medium    241
High       83
Name: count, dtype: int64

PK regime counts
regime_pk
Low       511
Medium    369
High       20
Name: count, dtype: int64

GK regime counts
regime_gk
Low       504
Medium    371
High       25
Name: count, dtype: int64
